# 04_01_手計算の逆伝播と自動微分の検算

コード：`ch04/blocks/04_01_手計算の逆伝播と自動微分の検算.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 4 章：2 層のネットの誤差逆伝播を手で 1 ステップ計算し、PyTorch の自動微分で検算する
import numpy as np, torch

x  = np.array([1.0, 2.0])                    # 入力 2 個
W1 = np.array([[0.1, -0.2], [0.4, 0.3]])     # 隠れ層 2 個（行が隠れユニット）
b1 = np.array([0.0, 0.1])
w2 = np.array([0.5, -0.6]); b2 = 0.2         # 出力 1 個
y  = 1.0                                     # 正解

# 順伝播
z1 = W1 @ x + b1                             # 線形変換
h  = np.maximum(z1, 0)                       # ReLU
yhat = w2 @ h + b2
L  = 0.5 * (yhat - y) ** 2                   # 二乗誤差
# 逆伝播：連鎖律を出力側から 1 段ずつ
dyhat = yhat - y                             # ∂L/∂ŷ
dw2, db2 = dyhat * h, dyhat                  # ∂L/∂w2 = ∂L/∂ŷ · h
dh  = dyhat * w2                             # ∂L/∂h
dz1 = dh * (z1 > 0)                          # ReLU の微分は 0 か 1
dW1, db1 = np.outer(dz1, x), dz1             # ∂L/∂W1 = ∂L/∂z1 · xᵀ
print(f"z1={z1}  h={h}  ŷ={yhat:.3f}  L={L:.4f}")
print(f"∂L/∂w2={dw2}  ∂L/∂b2={db2:.3f}")
print(f"∂L/∂W1=\n{dW1}")

# 自動微分で検算
t = lambda a: torch.tensor(a, dtype=torch.float64, requires_grad=True)
W1t, b1t, w2t, b2t = t(W1), t(b1), t(w2), t(b2)
Lt = 0.5 * (w2t @ torch.relu(W1t @ torch.tensor(x) + b1t) + b2t - y) ** 2
Lt.backward()
print("手計算と自動微分の最大差:",
      max(abs(W1t.grad.numpy() - dW1).max(), abs(w2t.grad.numpy() - dw2).max(),
          abs(b1t.grad.numpy() - db1).max(), abs(b2t.grad.item() - db2)))
# 1 ステップ更新して損失が下がるか
eta = 0.1
W1n, w2n, b1n, b2n = W1 - eta * dW1, w2 - eta * dw2, b1 - eta * db1, b2 - eta * db2
yhat2 = w2n @ np.maximum(W1n @ x + b1n, 0) + b2n
print(f"η=0.1 で 1 ステップ更新：L {L:.4f} → {0.5 * (yhat2 - y) ** 2:.4f}")

# 勾配消失：20 層のネットで、入力側の層ほど勾配がどれだけ小さくなるか
def layer_grads(act, he_init=False, depth=20, width=64, seed=0):
    torch.manual_seed(seed)
    layers = [torch.nn.Linear(width, width) for _ in range(depth)]
    if he_init:                               # He の初期化：分散 2/入力数 で重みを引く
        for l in layers:
            torch.nn.init.kaiming_normal_(l.weight, nonlinearity="relu")
            torch.nn.init.zeros_(l.bias)
    h = torch.randn(256, width)
    for l in layers:
        h = act(l(h))
    h.pow(2).mean().backward()
    return [l.weight.grad.norm().item() for l in layers]
for name, act, he in (("シグモイド", torch.sigmoid, False), ("ReLU", torch.relu, False),
                      ("ReLU＋He", torch.relu, True)):
    g = layer_grads(act, he)
    print(f"{name:6s} 勾配の大きさ：出力側の層 {g[-1]:.2e}  中間 {g[9]:.2e}  入力側の層 {g[0]:.2e}  比 {g[0] / g[-1]:.1e}")
